# 연습문제 1-5 풀이

이 노트북은 1-1절 연습문제 1-5(도전문제)의 풀이 예시다. 예시일 뿐 얼마든지 다른 구현을 통해 해결할 수도 있다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch

## 연습문제 1-5 [도전문제]

> 파이토치가 자동으로 수행하는 브로드캐스팅을 사용하지 않고, 3차원 리스트 `[[[1, 2]], [[3, 4]]]`와
> 2차원 리스트 `[[5, 6], [7, 8]]`로 만든 두 텐서의 덧셈 결과를 출력해 보자.
>
> 힌트: 브로드캐스팅 1단계는 차원을 추가하는 메서드를 사용하면 되며,
> 브로드캐스팅 2단계는 여러 텐서를 이어 붙이는 메서드를 사용하면 된다.

**1단계: 차원 맞추기**
- 텐서 `A`는 3차원 텐서, 텐서 `B`는 2차원 텐서
- 텐서 `B`의 앞쪽에 크기가 1인 차원을 하나 추가해 3차원 텐서로 확장한다.
    - 원래 텐서 `B`: `[[5, 6], [7, 8]]`   - `(2, 2)` 형태
    - 확장된 텐서  : `[[[5, 6], [7, 8]]]` - `(1, 2, 2)` 형태
- `unsqueeze()` 메서드 사용

In [2]:
A = torch.tensor([[[1, 2]], [[3, 4]]])    # (2, 1, 2) 형태
B = torch.tensor([[5, 6], [7, 8]])        # (2, 2) 형태
print(f'A: {A.shape}, B: {B.shape}')

# 1단계: 차원 수가 적은 B의 앞쪽에 크기가 1인 차원을 추가해 차원 수를 맞춘다
B_expanded = B.unsqueeze(0)               # (1, 2, 2)
print(f'1단계 후 B: {B_expanded.shape}')

A: torch.Size([2, 1, 2]), B: torch.Size([2, 2])
1단계 후 B: torch.Size([1, 2, 2])


**2단계: 형태 맞추기**
- 크기가 1인 차원을 상대 텐서의 크기에 맞춰 복사해 늘림
    - `torch.cat()` 함수로 복사할 횟수 만큼 자신을 반복해 이어 붙임
- 크기가 1이 아닌 차원의 크기가 서로 다르면 브로드캐스팅 불가
- 텐서 `A`와 텐서 `B`의 형태 변화
    - 텐서 `A`: `(2, 1, 2)` 형태 -> `(2, 2, 2)` 형태(두 번째 차원 복사)
    - 텐서 `B`: `(1, 2, 2)` 형태 -> `(2, 2, 2)` 형태(첫 번째 차원 복사)

In [3]:
# 2단계: 크기가 1인 차원을 상대 텐서의 크기에 맞춰 복사해 늘린다
#   A는 두 번째 차원(크기 1)을 2로, B는 첫 번째 차원(크기 1)을 2로
A_expanded = torch.cat([A, A], dim=1)                        # (2, 2, 2)
B_expanded = torch.cat([B_expanded, B_expanded], dim=0)      # (2, 2, 2)
print(f'2단계 후 A: {A_expanded.shape}, B: {B_expanded.shape}')


2단계 후 A: torch.Size([2, 2, 2]), B: torch.Size([2, 2, 2])


In [4]:
# 형태가 같으므로 브로드캐스팅 없이 요소별 덧셈 가능
result = A_expanded + B_expanded
print(result)

# 파이토치의 브로드캐스팅 결과와 비교
print(f'결과가 같은지 확인: {torch.equal(result, A + B)}')

tensor([[[ 6,  8],
         [ 8, 10]],

        [[ 8, 10],
         [10, 12]]])
결과가 같은지 확인: True


**일반화: 임의의 두 텐서에 적용하기**
- 위 과정을 임의의 두 텐서에 대해 수행하는 함수로 만든다.
- 1단계: 차원 수가 적은 텐서의 앞쪽에 `unsqueeze(0)`으로 크기가 1인 차원을 추가한다.
- 2단계: 차원마다 두 텐서의 크기를 비교한다.
    - 크기가 같으면 그대로 둔다.
    - 한쪽이 1이면 그 텐서를 상대 크기만큼 `torch.cat()`으로 이어 붙인다.
    - 둘 다 1이 아니면서 서로 다르면 브로드캐스팅이 불가능하므로 오류를 발생시킨다.

In [5]:
def manual_broadcast(A, B):
    """브로드캐스팅을 쓰지 않고 두 텐서를 같은 형태로 맞춰 반환한다. 불가능하면 ValueError 예외 발생"""
    # 1단계: 차원 수가 적은 텐서의 앞쪽에 크기가 1인 차원을 추가한다
    ndim = max(A.dim(), B.dim())
    while A.dim() < ndim:
        A = A.unsqueeze(0)
    while B.dim() < ndim:
        B = B.unsqueeze(0)

    # 2단계: 차원마다 크기를 비교해 크기가 1인 쪽을 상대 크기만큼 복사한다
    for d in range(ndim):
        a, b = A.shape[d], B.shape[d]
        if a == b:
            continue
        if a == 1:
            A = torch.cat([A] * b, dim=d)
        elif b == 1:
            B = torch.cat([B] * a, dim=d)
        else:
            raise ValueError(f'브로드캐스팅 불가: 뒤에서 {ndim - d}번째 차원의 크기가 {a}와 {b}로 다르고 둘 다 1이 아니다')
    return A, B


def manual_add(A, B):
    A_b, B_b = manual_broadcast(A, B)
    return A_b + B_b

In [6]:
# 연습문제의 텐서: 브로드캐스팅 가능
print(f'연습문제 결과가 같은지 확인: {torch.equal(manual_add(A, B), A + B)}')

# 여러 형태 조합으로 파이토치 결과와 비교
for shape_a, shape_b in [((3, 1), (1, 4)), ((5,), (2, 5)), ((2, 1, 3), (4, 1)), ((), (2, 2))]:
    a, b = torch.randint(0, 10, shape_a), torch.randint(0, 10, shape_b)
    print(f'{shape_a} + {shape_b} -> {tuple(manual_add(a, b).shape)}, '
          f'자동 브로드캐스팅 결과와 비교: {torch.equal(manual_add(a, b), a + b)}')

# 브로드캐스팅 불가: 마지막 차원의 크기가 3과 4로 다르고 둘 다 1이 아니다
try:
    manual_add(torch.zeros(2, 3), torch.zeros(2, 4))
except ValueError as e:
    print(e)

연습문제 결과가 같은지 확인: True
(3, 1) + (1, 4) -> (3, 4), 자동 브로드캐스팅 결과와 비교: True
(5,) + (2, 5) -> (2, 5), 자동 브로드캐스팅 결과와 비교: True
(2, 1, 3) + (4, 1) -> (2, 4, 3), 자동 브로드캐스팅 결과와 비교: True
() + (2, 2) -> (2, 2), 자동 브로드캐스팅 결과와 비교: True
브로드캐스팅 불가: 뒤에서 1번째 차원의 크기가 3와 4로 다르고 둘 다 1이 아니다
